In [2]:
import os
import zipfile
import subprocess
import time
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import storage
from google.api_core import exceptions
import shutil

# 1. 环境初始化
PROJECT_ID = "gen-lang-client-0371685655"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)

MODEL_NAME = "gemini-2.5-pro" 
# 如果 2.5-pro 报错，请切回 "gemini-1.5-pro-002"
model = GenerativeModel(MODEL_NAME)

# 2. 路径配置 (已根据需求修改)
BUCKET_NAME = 'xhs-humor-data'
SEARCH_KEYWORD = "脱口秀大咖" 
LOCAL_BASE_DIR = './talkshow_verbatim_content'
CLOUD_RESULT_PREFIX = 'talkshow_verbatim_content' # 云端输出目录前缀

os.makedirs(LOCAL_BASE_DIR, exist_ok=True)
# 注意：RESULT_BASE_DIR 现在实际上就是 LOCAL_BASE_DIR 下的子结构，或者我们可以单独定义一个结果目录
# 为了逻辑清晰，我们将结果直接存放在 LOCAL_BASE_DIR/subject_name 下
# 如果你想完全分离临时下载目录和结果目录，可以保持如下：
RESULT_BASE_DIR = LOCAL_BASE_DIR # 将结果直接保存在该目录下

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

# --- [断点续传]：预加载已完成列表 ---
print(f"正在同步云端 [{CLOUD_RESULT_PREFIX}] 下已完成的任务列表...")
# 获取云端所有已经存在的 txt 文件路径
existing_blobs = set([b.name for b in bucket.list_blobs(prefix=CLOUD_RESULT_PREFIX)])
print(f"云端已存在 {len(existing_blobs)} 个解析结果，这些将被跳过。\n")
# --------------------------------------------------

# 3. 获取目标文件并预扫描总量
all_blobs = storage_client.list_blobs(BUCKET_NAME)
# 筛选包含 "脱口秀大咖" 且是 zip 的文件
target_zips = [b for b in all_blobs if SEARCH_KEYWORD in b.name and b.name.endswith('.zip')]

print(f"正在预扫描视频总量以计算全局进度...")
total_videos = 0
task_list = []

for zip_blob in target_zips:
    temp_zip = os.path.join(LOCAL_BASE_DIR, "scan.zip")
    zip_blob.download_to_filename(temp_zip)
    with zipfile.ZipFile(temp_zip, 'r') as z:
        # 排除 macOS 系统生成的临时文件
        v_list = [f for f in z.namelist() if f.lower().endswith(('.mp4', '.mov', '.avi')) and not f.startswith('__MACOSX')]
        total_videos += len(v_list)
        task_list.append((zip_blob, v_list))
    os.remove(temp_zip)

print(f"共计待处理视频总数: {total_videos}\n")

# --- 定义带重试机制的请求函数 ---
def generate_with_retry(prompt_text, audio_bytes, max_retries=5):
    """
    尝试调用 Gemini API。遇到 429 限流时指数退避。
    """
    retry_count = 0
    wait_time = 10 
    
    while retry_count < max_retries:
        try:
            audio_part = Part.from_data(data=audio_bytes, mime_type="audio/mpeg")
            response = model.generate_content([prompt_text, audio_part])
            return response.text.strip()
        
        except exceptions.ResourceExhausted:
            print(f"  [!] 触发限流 (429)，等待 {wait_time} 秒后重试 (第 {retry_count+1}/{max_retries} 次)...")
            time.sleep(wait_time)
            retry_count += 1
            wait_time *= 2 
            
        except exceptions.InternalServerError:
            print(f"  [!] 服务器内部错误 (500)，等待 5 秒重试...")
            time.sleep(5)
            retry_count += 1
            
        except Exception as e:
            raise e
            
    raise Exception("重试次数耗尽，API 调用失败")
# --------------------------------------------------

# 4. 核心处理循环
processed_count = 0
skipped_count = 0

for zip_blob, video_files in task_list:
    # 从文件名提取主题，例如 "脱口秀大咖_李诞" -> "脱口秀大咖_李诞"
    subject_name = zip_blob.name.split('/')[-1].replace('.zip', '')
    
    # 本地解压和结果路径
    subject_dir = os.path.join(RESULT_BASE_DIR, subject_name)
    os.makedirs(subject_dir, exist_ok=True)
    
    # 下载并解压当前主题包
    local_zip = os.path.join(LOCAL_BASE_DIR, "current.zip")
    print(f"\n>>> 正在下载并解压主题: {subject_name}")
    zip_blob.download_to_filename(local_zip)
    
    try:
        with zipfile.ZipFile(local_zip, 'r') as z:
            z.extractall(subject_dir)
    except zipfile.BadZipFile:
        print(f"错误: 压缩包损坏 {subject_name}，跳过")
        if os.path.exists(local_zip): os.remove(local_zip)
        continue
    
    # 遍历处理该主题下的每一个视频
    for root, _, files in os.walk(subject_dir):
        for file in files:
            if file.lower().endswith(('.mp4', '.mov', '.avi')) and not file.startswith('._'):
                processed_count += 1
                video_path = os.path.join(root, file)
                audio_path = os.path.join(root, os.path.splitext(file)[0] + ".mp3")
                txt_name = f"{os.path.splitext(file)[0]}.txt"
                
                # 定义云端目标路径 (使用新的前缀)
                cloud_txt_path = f"{CLOUD_RESULT_PREFIX}/{subject_name}/{txt_name}"

                # --- 断点续传检测 ---
                if cloud_txt_path in existing_blobs:
                    print(f"[{processed_count}/{total_videos}] 跳过已存在: {file}")
                    skipped_count += 1
                    # 即使跳过，也清理本地视频以释放空间，因为TXT已经存在于云端
                    if os.path.exists(video_path): os.remove(video_path)
                    continue
                # ------------------------------
                
                percent = (processed_count / total_videos) * 100
                print(f"进度: [{percent:.2f}%] | 正在处理: {file}")
                
                try:
                    # 第一步：提取音频
                    subprocess.run([
                        'ffmpeg', '-i', video_path, 
                        '-vn', '-acodec', 'libmp3lame', '-q:a', '2', 
                        audio_path
                    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                    
                    # 第二步：调用 Gemini (Prompt 已调整为脱口秀逐字稿/分析模式)
                    prompt = """
                    逐字转录脱口秀原文，并在每个段子或笑点结束后强制换行分段。直接输出原文。
                    """
                    
                    with open(audio_path, "rb") as f_audio:
                        audio_data = f_audio.read()
                    
                    result_text = generate_with_retry(prompt, audio_data)
                    
                    # 第三步：生成 TXT (保留在本地)
                    txt_local_path = os.path.join(subject_dir, txt_name)
                    # 确保子目录存在（防止 zip 解压层级问题）
                    os.makedirs(os.path.dirname(txt_local_path), exist_ok=True)
                    
                    with open(txt_local_path, "w", encoding="utf-8") as f_txt:
                        f_txt.write(result_text)
                    
                    # 上传云端
                    bucket.blob(cloud_txt_path).upload_from_filename(txt_local_path)
                    
                    # 更新内存列表
                    existing_blobs.add(cloud_txt_path)
                    
                    # 第四步：清理本地 视频和音频 (保留 TXT)
                    if os.path.exists(video_path): os.remove(video_path)
                    if os.path.exists(audio_path): os.remove(audio_path)
                    
                except Exception as e:
                    print(f"处理文件 {file} 失败: {e}")
                    if os.path.exists(audio_path): os.remove(audio_path)

    # 清理当前压缩包缓存
    if os.path.exists(local_zip):
        os.remove(local_zip)
    
    # --- [关键修改]：不再删除整个文件夹 ---
    # 原代码的 shutil.rmtree(subject_dir) 已被移除
    # 这样 TXT 文件会保留在 ./talkshow_verbatim_temp/主题名/ 目录下
    print(f"主题 {subject_name} 处理完毕，TXT文件已保留在本地。")

print(f"\n--- 任务全部完成！共扫描 {processed_count}，实际处理 {processed_count - skipped_count}，跳过 {skipped_count} ---")

/opt/conda/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


正在同步云端 [talkshow_verbatim_content] 下已完成的任务列表...
云端已存在 0 个解析结果，这些将被跳过。

正在预扫描视频总量以计算全局进度...
共计待处理视频总数: 176


>>> 正在下载并解压主题: 小红书博主【脱口秀大咖】笔记批量下载_20260212212300
进度: [0.57%] | 正在处理: 赛文讲自己讨厌人工智能_1.mp4
进度: [1.14%] | 正在处理: 小雪讲中年姐姐的魅力_1.mp4
进度: [1.70%] | 正在处理: 韩大狗讲廉价酒店_1.mp4
进度: [2.27%] | 正在处理: 山河讲公共厕所的尴尬_1.mp4
进度: [2.84%] | 正在处理: 二哥讲消防员的工作_1.mp4
  [!] 触发限流 (429)，等待 10 秒后重试 (第 1/5 次)...
  [!] 触发限流 (429)，等待 20 秒后重试 (第 2/5 次)...
进度: [3.41%] | 正在处理: 徐志胜讲回来讲脱口秀了_1.mp4
进度: [3.98%] | 正在处理: 哈瑞讲自己的名字不被记住_1.mp4
进度: [4.55%] | 正在处理: 艾斯讲我展示的习惯_1.mp4
进度: [5.11%] | 正在处理: 鸡翅讲自己的长相_1.mp4
进度: [5.68%] | 正在处理: 安全出口组合模仿餐厅用餐_1.mp4
进度: [6.25%] | 正在处理: 火锅讲自己当警察的事情_1.mp4
进度: [6.82%] | 正在处理: 李文讲写论文只有笑点_1.mp4
进度: [7.39%] | 正在处理: 大国手讲挣钱和哲学_1.mp4
进度: [7.95%] | 正在处理: 张骏讲自己出国上学_1.mp4
进度: [8.52%] | 正在处理: 贾耗讲自己化妆_1.mp4
  [!] 触发限流 (429)，等待 10 秒后重试 (第 1/5 次)...
进度: [9.09%] | 正在处理: 小块讲跟徐志胜做生意赔钱_1.mp4
进度: [9.66%] | 正在处理: 周欣雨讲姐弟恋_1.mp4
进度: [10.23%] | 正在处理: 漫才兄弟讲家里很乱_1.mp4
进度: [10.80%] | 正在处理: 菜菜讲自己的家人_1.mp4
进度: [11.36%] | 正在处理: 小四